# The Problem With a Single Test Set

When a dataset is divided only into training and test sets, the test set should be used only for the final evaluation.

If we repeatedly check the test set while tuning the model, the test results start influencing our development decisions.

This means that information from the test set has indirectly leaked into the model development process.

As a result, the test score may no longer be an honest estimate of how the model will perform on completely unseen real-world data.

Therefore, we need a separate validation set for model development and tuning.


In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

iris = load_iris()

X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Test Accuracy:", accuracy)

# The Three-Way Split

A professional machine learning workflow divides the dataset into three parts:

* **Training Set:** Used to train the model and learn its parameters.
* **Validation Set:** Used during development to tune hyperparameters, compare models, and test different features.
* **Test Set:** Used only once at the end to provide the final performance estimate.

A common split is:

* 60% Training
* 20% Validation
* 20% Test

The most important rule is to keep the test set untouched until the final evaluation.


In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()

X = iris.data
y = iris.target

# First: separate the test set
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

# Second: split the remaining data
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25,
    random_state=42
)

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

# Creating a Three-Way Split in Scikit-learn

A three-way split can be created using two calls to `train_test_split()`.

First, we hold out 20% of the data as the final test set.

Then, we split the remaining 80% into training and validation data.

Using `test_size=0.25` in the second split gives:

* 75% of the remaining data for training
* 25% of the remaining data for validation

The final proportions are approximately:

* 60% Training
* 20% Validation
* 20% Test


In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25,
    random_state=42
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

# Why One Validation Set Can Still Mislead

A single validation set is better than tuning directly on the test set, but it can still be misleading.

If the validation set happens to contain an unusual or unrepresentative part of the dataset, our model decisions may depend on luck rather than a reliable signal.

This problem is especially important with small datasets.

Cross-validation solves this problem by using multiple validation folds instead of relying on one validation split.


In [ ]:
from sklearn.model_selection import train_test_split

X_train1, X_val1, y_train1, y_val1 = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_train2, X_val2, y_train2, y_val2 = train_test_split(
    X, y,
    test_size=0.2,
    random_state=100
)

print("First validation set:", len(X_val1))
print("Second validation set:", len(X_val2))

# Hands-On Lab: Building a Three-Way Split

In [2]:
import pandas as pd

data = {
    "Age": [18,19,20,21,22,20,23,19,24,21,18,22,20,23,19,24,21,20,22,23],
    "Income": [2500,3000,3500,2200,4000,2800,4500,2600,5000,3200,
               2300,3800,2900,4200,2700,4800,3100,2500,3900,4400],
    "StudyHours": [2,4,5,1,6,3,7,2,8,4,1,6,2,7,3,8,5,2,6,7],
    "City": ["Hebron","Nablus","Ramallah","Hebron","Ramallah",
             "Nablus","Ramallah","Hebron","Ramallah","Nablus",
             "Hebron","Ramallah","Nablus","Ramallah","Hebron",
             "Ramallah","Nablus","Hebron","Ramallah","Nablus"],
    "Passed": [0,1,1,0,1,0,1,0,1,1,0,1,0,1,0,1,1,0,1,1]
}

df = pd.DataFrame(data)

df.head()

,Age,Income,StudyHours,City,Passed
0,18,2500,2,Hebron,0
1,19,3000,4,Nablus,1
2,20,3500,5,Ramallah,1
3,21,2200,1,Hebron,0
4,22,4000,6,Ramallah,1


# Step 1: Prepare the Features and Target

The dataset contains information about students and whether they passed.

The target variable is `Passed`.

The input features are:

- `Age`
- `Income`
- `StudyHours`
- `City`

Since `City` is a categorical feature, we need to convert it into numerical values before training the model.

We will use one-hot encoding for the `City` column.

In [3]:
from sklearn.preprocessing import OneHotEncoder

# Separate features and target
X = df.drop("Passed", axis=1)
y = df["Passed"]

# Convert City into numerical columns
X = pd.get_dummies(X, columns=["City"], dtype=int)

print(X.head())
print("\nFeatures shape:", X.shape)
print("Target shape:", y.shape)

   Age  Income  StudyHours  City_Hebron  City_Nablus  City_Ramallah
0   18    2500           2            1            0              0
1   19    3000           4            0            1              0
2   20    3500           5            0            0              1
3   21    2200           1            1            0              0
4   22    4000           6            0            0              1

Features shape: (20, 6)
Target shape: (20,)


# Step 2: Create a 60/20/20 Train / Validation / Test Split

We divide the dataset into three sets:

- **Training set (60%)**: Used to train the model.
- **Validation set (20%)**: Used to tune the model.
- **Test set (20%)**: Used only for the final evaluation.

We first separate 20% of the data as the test set.

Then, we split the remaining 80% into 75% training and 25% validation.

This results in approximately 60% training, 20% validation, and 20% test data.

In [4]:
from sklearn.model_selection import train_test_split

# First split: 20% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Second split:
# 75% of the remaining 80% = 60% training
# 25% of the remaining 80% = 20% validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

Training set: (12, 6)
Validation set: (4, 6)
Test set: (4, 6)


# Step 3: Train and Tune Using the Validation Set

We will use a K-Nearest Neighbors (KNN) classifier.

The `n_neighbors` value is a hyperparameter that controls how many neighboring samples the model uses to make a prediction.

We will test different values of `n_neighbors` and use the validation set to choose the best value.

The test set will not be used during this tuning process.

In [5]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

k_values = [1, 3, 5]

best_k = None
best_val_score = 0

for k in k_values:

    model = KNeighborsClassifier(n_neighbors=k)

    # Train using training data only
    model.fit(X_train, y_train)

    # Evaluate using validation data
    y_val_pred = model.predict(X_val)

    val_score = accuracy_score(y_val, y_val_pred)

    print(f"k = {k}, Validation Accuracy = {val_score:.2f}")

    # Store the best hyperparameter
    if val_score > best_val_score:
        best_val_score = val_score
        best_k = k

print("\nBest k:", best_k)
print("Best validation accuracy:", best_val_score)

k = 1, Validation Accuracy = 1.00
k = 3, Validation Accuracy = 0.75
k = 5, Validation Accuracy = 1.00

Best k: 1
Best validation accuracy: 1.0


# Step 4: Evaluate the Final Model on the Test Set

After tuning the model using the validation set, we select the best value of `n_neighbors`.

We then create the final model using the selected value.

The final model is evaluated on the test set only once.

The test score gives us an estimate of how the model performs on unseen data.

In [6]:
# Create the final model using the best k
final_model = KNeighborsClassifier(n_neighbors=best_k)

# Train the final model
final_model.fit(X_train, y_train)

# Predict on the test set
y_test_pred = final_model.predict(X_test)

# Calculate final test accuracy
test_score = accuracy_score(y_test, y_test_pred)

print("Best k:", best_k)
print("Final Test Accuracy:", test_score)

Best k: 1
Final Test Accuracy: 1.0


# Step 5: Why We Should Not Tune Against the Test Set

The test set must remain untouched during model development.

If we repeatedly check the test score and change the model based on that score, our decisions become influenced by the test data.

This causes information leakage from the test set into the development process.

As a result, the final test score may not represent the model's true performance on completely new data.

The correct workflow is:

1. Train the model using the training set.
2. Tune the model using the validation set.
3. Select the final model.
4. Evaluate the final model on the test set once.

Therefore, the test set should only be used at the very end.